In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from RS.PreRun import PreRun, PostRun
from sklearn.linear_model import LinearRegression


# path to data
read_path = "../../../data_ds_project/parquet_cleaned_energy"
systems_cleaned = pd.read_csv("../../data/core/systems_cleaned.csv")
relevant_system_pairs = [(10, None), (50, None), (51, None)]

In [ ]:
naive_folder_str = './RS/naive_errors/'
naive_file_suffix = '_naive_errors.csv'
lin_folder_str = './RS/linreg_errors/'
lin_file_suffix = '_linreg_errors.csv'
lin_fourier_suffix = '_linreg_errors_fourier.csv'
prophet_folder_str = './RS/prophet_errors/'
prophet_file_suffix = '_prophet_errors.csv'
sarimax_folder_str = './RS/sarimax_errors/'
sarimax_file_suffix = '_sarimax_errors.csv'
xgboost_folder_str = './CEB/xgboost_results/'
xgboost_folder_suffix = '.csv'
lightgbm_folder_str = './CEB/lightgbm_results/'
lightgbm_folder_suffix = '.csv'

model_tuple = ('naive', 'lin_reg', 'lin_reg_with_fourier', 'prophet', 'sarimax', 'xgboost', 'lightgbm')
model_folder = (naive_folder_str, lin_folder_str, lin_folder_str, prophet_folder_str,
                sarimax_folder_str, xgboost_folder_str, lightgbm_folder_str)
model_suffix = (naive_file_suffix, lin_file_suffix, lin_fourier_suffix, prophet_file_suffix,
                sarimax_file_suffix, xgboost_folder_suffix, lightgbm_folder_suffix)
sarimax_col_names = [
    '2,0,0', '2,0,1', '3,0,0', '3,0,1'
]
xgboost_col_names = [
    '(31, 5, 0.1, 1.0, 0.8)', '(31, 7, 0.1, 0.8, 0.8)', '(31, 10, 0.1, 0.8, 0.8)'
]

lightgbm_col_names = [
    '(31, -1, 0.1, 100, 0.8, 0.8)',
    '(31, -1, 0.1, 100, 1.0, 0.8)'
]


In [ ]:
def read_results_summaries(system_id: int):
    my_cols = ['per_model_mean', 'per_model_std', 'per_model_min', 'per_model_25p', 'per_model_median', 'per_model_75p', 'per_model_max']
    per_model_results = []
    for j in range(7):
        model_name = model_tuple[j]
        model_results = pd.read_csv(f'{model_folder[j]}{system_id}_None{model_suffix[j]}')
        if model_name == 'xgboost':
            model_results = model_results[xgboost_col_names]
        elif model_name == 'lightgbm':
            model_results = model_results[lightgbm_col_names]
        elif model_name == 'sarimax':
            model_results = model_results[sarimax_col_names]
        model_results = model_results.rename(columns={
            col_name: f'{model_name}_{col_name}' for col_name in model_results.columns
        })
        model_results = model_results.transpose()
        ordinary_cols = model_results.columns
        model_results.loc[:, 'per_model_mean'] = model_results[ordinary_cols].mean(axis=1)
        model_results.loc[:, 'per_model_std'] = model_results[ordinary_cols].std(axis=1)
        model_results.loc[:, 'per_model_min'] = model_results[ordinary_cols].min(axis=1)
        model_results.loc[:, 'per_model_25p'] = model_results[ordinary_cols].quantile(q=0.25, axis=1)
        model_results.loc[:, 'per_model_median'] = model_results[ordinary_cols].quantile(q=0.5, axis=1)
        model_results.loc[:, 'per_model_75p'] = model_results[ordinary_cols].quantile(q=0.75, axis=1)
        model_results.loc[:, 'per_model_max'] = model_results[ordinary_cols].max(axis=1)
        per_model_results.append(model_results[my_cols])
    total_results = pd.concat(per_model_results)
    return total_results


In [ ]:
read_results_summaries(10)

,per_model_mean,per_model_std,per_model_min,per_model_25p,per_model_median,per_model_75p,per_model_max
naive_error,0.044598,0.057223,0.000106,0.008121,0.020224,0.055983,0.323554
lin_reg_error,0.033743,0.043212,0.000505,0.006897,0.018895,0.044661,0.370300
lin_reg_with_fourier_error,0.033711,0.043174,0.000347,0.006709,0.018773,0.045067,0.369017
"prophet_(0.1, 10, 20, 2)",0.029864,0.034978,0.001236,0.008429,0.015200,0.037130,0.182796
"prophet_(0.5, 10, 20, 2)",0.030024,0.035139,0.001230,0.008471,0.015158,0.038870,0.181705
"sarimax_2,0,0",0.020586,0.031376,0.000259,0.004299,0.007294,0.029014,0.133287
"sarimax_2,0,1",0.020944,0.032378,0.000240,0.003970,0.007286,0.028132,0.135753
"sarimax_3,0,0",0.020698,0.031711,0.000240,0.004371,0.007316,0.028960,0.135702
"sarimax_3,0,1",0.021055,0.031653,0.000240,0.004348,0.007284,0.028961,0.135757
"xgboost_(31, 5, 0.1, 1.0, 0.8)",0.038852,0.038669,0.000292,0.013059,0.025964,0.052271,0.262199


In [ ]:
read_results_summaries(50)

,per_model_mean,per_model_std,per_model_min,per_model_25p,per_model_median,per_model_75p,per_model_max
naive_error,1.747726,2.641483,2.944007e-03,0.249494,0.630318,1.822918,16.621731
lin_reg_error,1.198927,1.607186,1.932127e-02,0.265745,0.643737,1.444014,14.255237
lin_reg_with_fourier_error,1.190300,1.608457,1.598518e-02,0.252537,0.636493,1.447720,14.219866
"prophet_(0.1, 10, 20, 2)",1.253558,1.567298,1.623266e-02,0.303753,0.633534,1.334442,7.453973
"sarimax_2,0,0",0.427444,0.581676,9.573573e-12,0.110832,0.244722,0.502947,4.652707
"sarimax_2,0,1",0.436378,0.587891,9.573573e-12,0.112829,0.252629,0.521132,4.634019
"sarimax_3,0,0",0.423098,0.578569,9.573573e-12,0.109676,0.227699,0.495708,4.626921
"sarimax_3,0,1",0.419532,0.579732,9.573573e-12,0.109138,0.222265,0.496839,4.636144
"xgboost_(31, 5, 0.1, 1.0, 0.8)",1.053864,1.212296,3.277221e-03,0.295317,0.657174,1.363442,8.290764
"xgboost_(31, 7, 0.1, 0.8, 0.8)",1.041629,1.206276,2.204182e-03,0.301800,0.648082,1.435311,8.887016


In [ ]:
read_results_summaries(51)

,per_model_mean,per_model_std,per_model_min,per_model_25p,per_model_median,per_model_75p,per_model_max
naive_error,1.483751,2.052073,0.001444,0.240178,0.614046,1.778395,15.571347
lin_reg_error,1.102705,1.460331,0.029086,0.244206,0.590222,1.413839,12.430139
lin_reg_with_fourier_error,1.098716,1.460753,0.028359,0.225830,0.593590,1.430833,12.529982
"prophet_(0.1, 10, 20, 2)",0.960666,1.030279,0.061602,0.285210,0.545052,1.299444,5.326777
"sarimax_2,0,0",0.527453,0.725216,0.011611,0.137603,0.281437,0.556825,4.842274
"sarimax_2,0,1",0.533469,0.718289,0.022000,0.137649,0.282606,0.548715,4.815697
"sarimax_3,0,0",0.517999,0.722429,0.011896,0.135268,0.276840,0.498896,4.805762
"sarimax_3,0,1",0.517141,0.724700,0.011885,0.134281,0.277228,0.499004,4.817465
"xgboost_(31, 5, 0.1, 1.0, 0.8)",1.086789,1.011577,0.012522,0.426061,0.780459,1.374913,6.236039
"xgboost_(31, 7, 0.1, 0.8, 0.8)",1.100136,1.035525,0.019361,0.438710,0.756980,1.378442,6.097769


OK, Sarimax appears to be the best model-set in training!